### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [15]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [16]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [6]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [7]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [8]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [9]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [10]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [11]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [12]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are several reasons in favor of using AutoGen in your new AI Agent project:

1. **Scalability**: AutoGen offers a modular and extensible framework, allowing you to create scalable systems that can grow and adapt with your project's needs.

2. **Ease of Use**: The integrated observability and debugging tools enhance the monitoring and control of agent workflows, making it easier to analyze performance and troubleshoot issues.

3. **Customization**: The framework allows for customizable systems tailored to specific project requirements, which can enhance functionality and user experience.

4. **Efficient Workflow Management**: AutoGen can effectively orchestrate interactions between multiple agents, streamlining processes and improving collaboration among them.

5. **Rapid Development**: AutoGen accelerates the development process by providing pre-built components and simplifying complex integrations, allowing you to deploy your AI agents faster.

Overall, AutoGen combines flexibility, efficiency, and user-friendliness, making it a strong candidate for AI Agent projects. 

TERMINATE

## Cons of AutoGen:
Here are some reasons against choosing AutoGen for a new AI Agent project:

1. **Poor Documentation**: The documentation for AutoGen is considered hard to read and lacks sufficient examples, which can hinder development and troubleshooting.

2. **Functional Limitations**: Certain features, such as structured outputs, may not work reliably, potentially complicating implementation.

3. **Lack of Differentiation**: AutoGen may not adequately differentiate itself from similar applications (like AG2), which could impact its competitiveness and long-term viability.

4. **Complexity of Use**: The design and functionality may be complex and not intuitive, leading to a steeper learning curve for developers.

These factors could be detrimental to the adoption of AutoGen in your AI project. 

TERMINATE



## Decision:

Based on the analysis of the pros and cons, I recommend using AutoGen for the project. The advantages, such as scalability, ease of use, customization, efficient workflow management, and rapid development, present a compelling case for its adoption. Although the cons, especially poor documentation and some functional limitations, are concerning, the overall benefits of flexibility and efficiency in managing AI agents outweigh these drawbacks. Effective orchestration of multiple agents can significantly enhance project outcomes. 

TERMINATE

In [13]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [14]:
await host.stop()